In [14]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from docx import Document, document
import gradio as ui

load_dotenv(override=True)

Base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv("GOOGLE_API_KEY")
Model = "gemini-3.6-flash"

myAi = OpenAI(base_url=Base_url,api_key=api_key)

system_message = """
You are AI Study Buddy, a helpful and intelligent study assistant.

Your main job is to help students understand and learn from their uploaded
documents such as PDF, DOCX, and TXT files.

Follow these rules:

1. Use the uploaded document as the primary source of information.
2. When answering questions about the document, stay faithful to its content.
3. If the answer is not available in the uploaded document, clearly say:
   "I couldn't find this information in the uploaded document."
   Do not make up information.
4. Explain difficult concepts in simple and beginner-friendly language.
5. Give examples whenever they help the student understand a concept.
6. When asked to summarize, provide concise and well-structured notes.
7. When asked for MCQs, generate clear questions with 4 options and provide
   the correct answer separately.
8. When asked for exam questions, create questions based on the uploaded
   study material.
9. When asked to explain code, explain it step-by-step and mention important
   concepts, errors, and time/space complexity when relevant.
10. If the student asks a follow-up question, use the previous conversation
    context to maintain continuity.
11. Do not unnecessarily make answers complicated.
12. Use headings, bullet points, numbered lists, and code blocks when useful.
13. Encourage learning by explaining the reasoning instead of simply giving
    an answer when appropriate.
14. If the user asks something unrelated to the uploaded document, you may
    answer using your general knowledge, but clearly distinguish it from
    information found in the document.

Your tone should be friendly, encouraging, clear, and student-friendly.

You are not just an answer generator. Your goal is to help the student
understand, revise, practice, and learn effectively.
"""

def read_file(file):
    if file is None:
        return 'Please upload a file'
    
    ext = os.path.splitext(file)[1].lower()

    if ext == '.pdf':
        reader = PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    
    elif ext == '.docx':
        doc = Document(file)
        text = ''
        for paragraph in doc.paragraphs:
            text+=paragraph.text +'\n'
        return text
    
    elif ext == '.txt':
        text = ''
        with open(file,'r',encoding="utf-8") as file:
            text = file.read() 
        return text

    else:
        return "Unsupported File Format"

def chat(message,history,file):
    text = read_file(file)
    messages = [{'role':'system','content':system_message}]+history+[{'role':'user','content':text+message}]
    result = myAi.chat.completions.create(
        model=Model,
        messages=messages,
        stream = True
    )
    response = ''
    for chunks in result:
        rim = chunks.choices[0].delta.content or ''
        response += rim
        yield response

file = ui.File(
        label="📚 Upload your study material",
        file_types=[".pdf", ".docx", ".txt"],
        type="filepath"
    )
    
ui.ChatInterface(fn= chat,
    save_history=True,
    additional_inputs=[file],
    title="STUDY VERSE AI",
    description="""
        # 🧠✨ StudyVerse AI
        ### *Turn your notes into an intelligent learning experience.*

        📚 **Upload • Ask • Understand • Practice • Master**

        Upload your **PDF, DOCX, or TXT** files and let AI help you
        understand concepts, summarize notes, generate MCQs, and prepare
        for exams.
        """
        ).launch(share=True,inline=False,inbrowser=True)


* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://c0bd24f8d6528e8362.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
import gradio as gr

with gr.Blocks(title="AI Study Buddy") as demo:

    gr.Markdown(
        """
        # 🤖 AI Study Buddy

        **Your personal AI-powered study assistant**

        Upload your **PDF, DOCX, or TXT** study material and ask questions,
        generate summaries, create MCQs, and understand difficult concepts
        with the help of Gemini AI.
        """
    )

    # Your components will come here
    # file = gr.File(...)
    # chatbot = gr.Chatbot(...)
    # textbox = gr.Textbox(...)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
